# Dask - Introduction: Solutions

This notebook contains solutions for all 7 exercises from the Dask lab.

## Shared Setup

Run this cell once before any exercise.

In [ ]:
import dask.dataframe as dd
import dask.bag as db
import os
import json
import dask
import time

from dask.distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
print(cluster.dashboard_link)

---

## Exercise 1 — Dask Bags: People Dataset

**Questions:**
1. Generate the random people dataset
2. What is the average age of people?
3. Which is the most common name?
4. What is the average age for each profession?
5. How many credit cards expire in 2025?

In [ ]:
# 1. Generate the random people dataset and write to disk
os.makedirs('data_ex1', exist_ok=True)

b = dask.datasets.make_people(npartitions=100, records_per_partition=10000)
b.map(json.dumps).to_textfiles('data_ex1/*.json')

In [ ]:
# Load the data back as a Bag and convert to a persisted DataFrame
people_bag = db.read_text('data_ex1/*.json').map(json.loads)

people_df = people_bag.to_dataframe().persist()
people_df.head()

In [ ]:
# 2. Average age of people
avg_age = people_df["age"].mean().compute()
print(f"Average age: {avg_age:.2f}")

In [ ]:
# 3. Most common first name
# The 'name' field is a list [first_name, last_name]; we extract the first element
name_frequencies = people_bag.map(lambda x: x["name"][0]).frequencies().compute()
most_common_name = max(name_frequencies, key=lambda x: x[1])
print(f"Most common name: '{most_common_name[0]}' (appears {most_common_name[1]} times)")

In [ ]:
# 4. Average age for each profession (occupation)
avg_age_by_profession = (
    people_df
    .drop("name", axis=1)
    .groupby("occupation")
    .agg({"age": "mean"})
    .compute()
    .sort_values("age", ascending=False)
)
avg_age_by_profession

In [ ]:
# 5. How many credit cards expire in 2025?
# The expiration-date format is "MM/YY"; we check if the year part is "25"
expiring_2025 = (
    people_bag
    .map(lambda x: x["credit-card"]["expiration-date"].split("/")[1])
    .filter(lambda year: year == "25")
    .count()
    .compute()
)
print(f"Credit cards expiring in 2025: {expiring_2025}")

---

## Exercise 2 — NYC Green Taxi CSV (GCS)

**Questions:**
1. Load the green taxi data from GCS: `gcs://anaconda-public-data/nyc-taxi/csv/2015/green_tripdata_*.csv`
2. Find the minimum number of passengers
3. Find the maximum number of passengers
4. Find the maximum cost for each number of passengers

In [ ]:
# 1. Load the green taxi data from GCS
# Note: gcsfs must be installed: pip install gcsfs
green_df = dd.read_csv(
    'gcs://anaconda-public-data/nyc-taxi/csv/2015/green_tripdata_*.csv',
    storage_options={'token': 'anon'},
    dtype={
        'Tolls_amount': 'float64',
        'Trip_type ': 'float64',   # note the trailing space in the column name
    }
)

# green_df = green_df.persist()
green_df.head()

In [ ]:
# 2. Minimum number of passengers
min_passengers = green_df['Passenger_count'].min().compute()
print(f"Minimum number of passengers: {min_passengers}")

In [ ]:
# 3. Maximum number of passengers
max_passengers = green_df['Passenger_count'].max().compute()
print(f"Maximum number of passengers: {max_passengers}")

In [ ]:
# 4. Maximum total cost for each number of passengers
max_cost_per_passenger = (
    green_df
    .groupby('Passenger_count')['Total_amount']
    .max()
    .compute()
    .sort_index()
)
max_cost_per_passenger

---

## Exercise 3 — NYC Yellow Taxi CSV (GCS)

**Questions:**
1. Load the yellow taxi data from GCS: `gcs://anaconda-public-data/nyc-taxi/csv/2015/yellow_*.csv`
2. Find the vendor with the highest number of trips
3. Find the vendor with the highest tip
4. Find the vendor that made the most money and how much
5. Find the weekday with the highest tip
6. Find the weekday with the highest number of runs

In [ ]:
# 1. Load the yellow taxi CSV data from GCS
yellow_csv = dd.read_csv(
    'gcs://anaconda-public-data/nyc-taxi/csv/2015/yellow_*.csv',
    storage_options={'token': 'anon'},
    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'],
    dtype={'tolls_amount': 'float64'}
)

yellow_csv = yellow_csv.persist()
yellow_csv.head()

In [ ]:
# 2. Vendor with the highest number of trips
trip_counts = yellow_csv.groupby('VendorID').size().compute()
best_vendor_trips = trip_counts.idxmax()
print(f"Vendor with most trips: VendorID {best_vendor_trips} ({trip_counts[best_vendor_trips]:,} trips)")
trip_counts

In [ ]:
# 3. Vendor with the highest total tip
tip_by_vendor = yellow_csv.groupby('VendorID')['tip_amount'].sum().compute()
best_vendor_tip = tip_by_vendor.idxmax()
print(f"Vendor with highest total tip: VendorID {best_vendor_tip} (${tip_by_vendor[best_vendor_tip]:,.2f})")
tip_by_vendor

In [ ]:
# 4. Vendor with the most revenue (total_amount)
revenue_by_vendor = yellow_csv.groupby('VendorID')['total_amount'].sum().compute()
best_vendor_revenue = revenue_by_vendor.idxmax()
print(f"Vendor with most revenue: VendorID {best_vendor_revenue} (${revenue_by_vendor[best_vendor_revenue]:,.2f})")
revenue_by_vendor

In [ ]:
# 5 & 6. Weekday analysis
# Extract day of week from pickup datetime (0=Monday, 6=Sunday)
yellow_csv_wd = yellow_csv.assign(weekday=yellow_csv['tpep_pickup_datetime'].dt.dayofweek)

day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}

# 5. Weekday with highest average tip
tip_by_day = yellow_csv_wd.groupby('weekday')['tip_amount'].mean().compute()
best_day_tip = tip_by_day.idxmax()
print(f"Weekday with highest average tip: {day_names[best_day_tip]} (avg ${tip_by_day[best_day_tip]:.2f})")
print(tip_by_day.rename(day_names))

In [ ]:
# 6. Weekday with the highest number of runs (trips)
runs_by_day = yellow_csv_wd.groupby('weekday').size().compute()
best_day_runs = runs_by_day.idxmax()
print(f"Weekday with most runs: {day_names[best_day_runs]} ({runs_by_day[best_day_runs]:,} trips)")
print(runs_by_day.rename(day_names))

In [ ]:
del yellow_csv

---

## Exercise 4 — NYC Yellow Taxi Parquet (GCS) + Performance Comparison

Same questions as Ex 3 but using the Parquet version of the dataset.

**Questions 2–6:** Same as Exercise 3  
**Question 7:** Is Parquet faster than CSV?

In [ ]:
# 1. Load the yellow taxi data from GCS as Parquet
yellow_parquet = dd.read_parquet(
    'gcs://anaconda-public-data/nyc-taxi/2015.parquet',
    storage_options={'token': 'anon'}
)

yellow_parquet = yellow_parquet.persist()
yellow_parquet.head()

In [ ]:
yellow_parquet.dtypes

In [ ]:
# 2. Vendor with the highest number of trips (Parquet)
trip_counts_pq = yellow_parquet.groupby('VendorID').size().compute()
best_vendor_trips_pq = trip_counts_pq.idxmax()
print(f"Vendor with most trips: VendorID {best_vendor_trips_pq} ({trip_counts_pq[best_vendor_trips_pq]:,} trips)")
trip_counts_pq

In [ ]:
# 3. Vendor with the highest total tip (Parquet)
tip_by_vendor_pq = yellow_parquet.groupby('VendorID')['tip_amount'].sum().compute()
best_vendor_tip_pq = tip_by_vendor_pq.idxmax()
print(f"Vendor with highest total tip: VendorID {best_vendor_tip_pq} (${tip_by_vendor_pq[best_vendor_tip_pq]:,.2f})")
tip_by_vendor_pq

In [ ]:
# 4. Vendor with the most revenue (Parquet)
revenue_by_vendor_pq = yellow_parquet.groupby('VendorID')['total_amount'].sum().compute()
best_vendor_revenue_pq = revenue_by_vendor_pq.idxmax()
print(f"Vendor with most revenue: VendorID {best_vendor_revenue_pq} (${revenue_by_vendor_pq[best_vendor_revenue_pq]:,.2f})")
revenue_by_vendor_pq

In [ ]:
# 5 & 6. Weekday analysis (Parquet)
# The parquet dataset uses 'tpep_pickup_datetime' — check dtypes above to confirm column name
yellow_parquet_wd = yellow_parquet.assign(
    weekday=yellow_parquet['tpep_pickup_datetime'].dt.dayofweek
)

day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}

# 5. Weekday with highest average tip
tip_by_day_pq = yellow_parquet_wd.groupby('weekday')['tip_amount'].mean().compute()
best_day_tip_pq = tip_by_day_pq.idxmax()
print(f"Weekday with highest average tip: {day_names[best_day_tip_pq]} (avg ${tip_by_day_pq[best_day_tip_pq]:.2f})")
print(tip_by_day_pq.rename(day_names))

In [ ]:
# 6. Weekday with the highest number of runs (Parquet)
runs_by_day_pq = yellow_parquet_wd.groupby('weekday').size().compute()
best_day_runs_pq = runs_by_day_pq.idxmax()
print(f"Weekday with most runs: {day_names[best_day_runs_pq]} ({runs_by_day_pq[best_day_runs_pq]:,} trips)")
print(runs_by_day_pq.rename(day_names))

In [ ]:
# 7. Performance comparison: CSV vs Parquet
# Time the vendor trip count query on CSV
start_csv = time.time()
yellow_csv.groupby('VendorID').size().compute()
csv_time = time.time() - start_csv

# Time the same query on Parquet
start_pq = time.time()
yellow_parquet.groupby('VendorID').size().compute()
pq_time = time.time() - start_pq

print(f"CSV time:     {csv_time:.2f}s")
print(f"Parquet time: {pq_time:.2f}s")
print(f"Parquet is {csv_time / pq_time:.1f}x faster")

**Why is Parquet faster than CSV?**

- **Columnar storage**: Parquet stores data column-by-column. When a query only needs a few columns (e.g., `VendorID`, `tip_amount`), Parquet reads only those columns from disk — CSV must scan every row in full.
- **Predicate pushdown**: Parquet row groups store min/max statistics, allowing the engine to skip entire chunks of data that cannot match a filter.
- **No schema parsing**: CSV requires inferring types on every read; Parquet stores schema metadata explicitly, so no guessing or type coercion is needed.
- **Compression**: Parquet supports efficient columnar compression (Snappy, Gzip), reducing I/O.

---

## Exercise 5 — Pokemon Dataset (Local Files)

**Prerequisites:** Download the two files from the course Drive and place them in a `data_ex5/` folder:
- `pockemonDB_dataset.csv` — Pokemon species specifications
- `trainers_with_pockemon.parquet` — Trainers and their Pokemon teams (unzip the `.zip` first)

**Questions:**
3. Who is the trainer with the highest average Defense in their team?
4. Who is the youngest trainer with a Fire-type Pokemon?
5. Who is the trainer with the most Water-type Pokemon?
6. How many coaches have a Psyduck on their team?
7. How many trainers are carrying an illegal number of Pokemon (more than 6)?

In [ ]:
# NOTE: adjust these paths to where you downloaded and unzipped the files
POKEMON_CSV   = 'data_ex5/pockemonDB_dataset.csv'
TRAINERS_PARQUET = 'data_ex5/trainers_with_pockemon.parquet'

# Load Pokemon species data
pokemon_df = dd.read_csv(POKEMON_CSV)
pokemon_df.head()

In [ ]:
# Load trainer data
trainers_df = dd.read_parquet(TRAINERS_PARQUET)
trainers_df.head()

In [ ]:
trainers_df.dtypes

In [ ]:
# Merge: join trainers with pokemon specs on the pokemon name
# Adjust the join keys below based on the actual column names shown above
merged = trainers_df.merge(pokemon_df, left_on='pokemon_name', right_on='Name')
merged = merged.persist()
merged.head()

In [ ]:
# 3. Trainer with the highest average Defense in their team
avg_defense = (
    merged
    .groupby('trainer_name')['Defense']
    .mean()
    .compute()
)
best_defender = avg_defense.idxmax()
print(f"Trainer with highest avg Defense: {best_defender} ({avg_defense[best_defender]:.2f})")

In [ ]:
# 4. Youngest trainer with a Fire-type Pokemon
# Filter to trainers who have at least one Fire-type pokemon
fire_trainers = merged[merged['Type 1'] == 'Fire']

# Find the minimum age among those trainers and retrieve the trainer name
youngest = (
    fire_trainers
    .groupby('trainer_name')['trainer_age']
    .min()
    .compute()
)
youngest_trainer = youngest.idxmin()
print(f"Youngest trainer with Fire-type Pokemon: {youngest_trainer} (age {youngest[youngest_trainer]})")

In [ ]:
# 5. Trainer with the most Water-type Pokemon
water_counts = (
    merged[merged['Type 1'] == 'Water']
    .groupby('trainer_name')
    .size()
    .compute()
)
best_water_trainer = water_counts.idxmax()
print(f"Trainer with most Water-type Pokemon: {best_water_trainer} ({water_counts[best_water_trainer]} Water Pokemon)")

In [ ]:
# 6. How many coaches have a Psyduck on their team?
psyduck_coaches = (
    merged[merged['Name'] == 'Psyduck']['trainer_name']
    .nunique()
    .compute()
)
print(f"Coaches with Psyduck: {psyduck_coaches}")

In [ ]:
# 7. How many trainers carry an illegal team (more than 6 Pokemon)?
team_sizes = (
    merged
    .groupby('trainer_name')
    .size()
    .compute()
)
illegal_trainers = (team_sizes > 6).sum()
print(f"Trainers with illegal team size (>6): {illegal_trainers}")

---

## Exercise 6 — Yelp Dataset (Local Files)

**Prerequisites:** Download the Yelp dataset from Kaggle and unzip it to a local folder.  
Expected files inside the unzipped directory:
- `yelp_academic_dataset_user.json`
- `yelp_academic_dataset_business.json`

**Questions:**
2. Who is the user with the highest number of friends?
3. How many businesses are located in "Santa Barbara"?
4. Export a Parquet file with columns: City, Number_of_businesses

In [ ]:
# NOTE: adjust this path to where you unzipped the Yelp dataset
YELP_DIR = 'data_ex6/'

# Load user and business data (each file is newline-delimited JSON)
users_df = dd.read_json(YELP_DIR + 'yelp_academic_dataset_user.json', lines=True)
business_df = dd.read_json(YELP_DIR + 'yelp_academic_dataset_business.json', lines=True)

users_df.head()

In [ ]:
business_df.head()

In [ ]:
# 2. User with the highest number of friends
# The 'friends' field is a comma-separated string of user IDs
# Count friends by splitting on ', ' and taking the length
users_with_count = users_df.assign(
    friend_count=users_df['friends'].str.split(', ').map(len, meta=('friends', 'int64'))
)

top_user = (
    users_with_count
    .nlargest(1, 'friend_count')[['name', 'friend_count']]
    .compute()
)
print(f"User with most friends: {top_user['name'].values[0]} ({top_user['friend_count'].values[0]:,} friends)")

In [ ]:
# 3. Number of businesses in Santa Barbara
santa_barbara_count = (
    business_df[business_df['city'] == 'Santa Barbara']
    .shape[0]
    .compute()           # shape[0] on a Dask DataFrame returns a delayed scalar
)
print(f"Businesses in Santa Barbara: {santa_barbara_count}")

In [ ]:
# 4. Export a Parquet with City and Number_of_businesses
city_businesses = (
    business_df
    .groupby('city')
    .size()
    .reset_index()
    .rename(columns={'city': 'City', 0: 'Number_of_businesses'})
    .compute()
)

city_businesses.to_parquet('city_businesses.parquet', index=False)
print("Exported city_businesses.parquet")
city_businesses.sort_values('Number_of_businesses', ascending=False).head(10)

---

## Exercise 7 — Yelp Reviews Analysis

**Prerequisites:** Same Yelp dataset as Exercise 6, plus:
- `yelp_academic_dataset_review.json`

**Questions:**
2. Compute the average number of stars per user across all reviews
3. Does the `average_stars` parameter in the user dataset match the computed average (within a tolerance)? Export mismatches.
4. Export a Parquet with columns: Name, Avg_rating, Avg_words
5. How many users have an Avg_rating < 3?

In [ ]:
# Load the reviews dataset (same YELP_DIR as Ex 6)
reviews_df = dd.read_json(YELP_DIR + 'yelp_academic_dataset_review.json', lines=True)
reviews_df.head()

In [ ]:
# 2. Compute average stars per user from reviews
avg_stars_from_reviews = (
    reviews_df
    .groupby('user_id')['stars']
    .mean()
    .compute()
    .reset_index()
    .rename(columns={'stars': 'avg_stars_computed'})
)
avg_stars_from_reviews.head()

In [ ]:
# 3. Compare computed average with the 'average_stars' field in the user dataset
# Tolerance of 0.1 stars is "reasonable" given rounding in the stored value
TOLERANCE = 0.1

users_computed = (
    users_df[['user_id', 'name', 'average_stars']]
    .compute()
    .merge(avg_stars_from_reviews, on='user_id')
)

mismatches = users_computed[
    (users_computed['average_stars'] - users_computed['avg_stars_computed']).abs() > TOLERANCE
]

print(f"Users where average_stars does not match (tolerance={TOLERANCE}): {len(mismatches):,}")

# Export mismatches
mismatches.to_parquet('mismatched_users.parquet', index=False)
print("Exported mismatched_users.parquet")
mismatches.head()

In [ ]:
# 4. Export a Parquet with Name, Avg_rating, Avg_words
# Compute average word count per user from review text
avg_words_from_reviews = (
    reviews_df
    .assign(word_count=reviews_df['text'].str.split().map(len, meta=('text', 'int64')))
    .groupby('user_id')['word_count']
    .mean()
    .compute()
    .reset_index()
    .rename(columns={'word_count': 'Avg_words'})
)

# Build the final export table
user_stats = (
    users_df[['user_id', 'name']]
    .compute()
    .merge(avg_stars_from_reviews.rename(columns={'avg_stars_computed': 'Avg_rating'}), on='user_id')
    .merge(avg_words_from_reviews, on='user_id')
    .rename(columns={'name': 'Name'})
    [['Name', 'Avg_rating', 'Avg_words']]
)

user_stats.to_parquet('user_stats.parquet', index=False)
print("Exported user_stats.parquet")
user_stats.head()

In [ ]:
# 5. How many users have an Avg_rating < 3?
low_rating_users = (user_stats['Avg_rating'] < 3).sum()
print(f"Users with Avg_rating < 3: {low_rating_users:,}")